# 02 — QC and Filtering

Per-sample QC, mitochondrial % filtering, MAD-based outlier removal, Scrublet doublet detection.

**Input**: `data/processed/01_raw.h5ad`  
**Output**: `data/processed/02_qc_filtered.h5ad`

Parameters come from `src/config.py`. If you want to change thresholds, edit the config, not this notebook.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import scanpy as sc
import matplotlib.pyplot as plt

from src import config as cfg
from src import qc as qc

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(4, 4))

adata = sc.read_h5ad(cfg.H5AD_RAW)
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')

## Compute QC metrics

In [ ]:
qc.annotate_qc_metrics(adata)
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo']].describe().round(2)

In [ ]:
# Per-sample QC summary BEFORE filtering
qc.summarize_qc(adata).head(30)

In [ ]:
# Visualize QC distributions
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(1, 3, figsize=(14, 4))

# Use seaborn violin without jitter dots; far cleaner for large data
sns.violinplot(y=adata.obs['n_genes_by_counts'], ax=axs[0], color='steelblue', inner='quartile')
axs[0].set_title('Genes per cell')
axs[0].set_yscale('log')   # log scale for skewed data
axs[0].set_ylabel('n_genes_by_counts (log)')

sns.violinplot(y=adata.obs['total_counts'], ax=axs[1], color='steelblue', inner='quartile')
axs[1].set_title('UMI counts per cell')
axs[1].set_yscale('log')
axs[1].set_ylabel('total_counts (log)')

sns.violinplot(y=adata.obs['pct_counts_mt'], ax=axs[2], color='steelblue', inner='quartile')
axs[2].set_title('Mitochondrial %')
axs[2].axhline(20, color='red', linestyle='--', label='Wu cutoff (20%)')
axs[2].legend()
axs[2].set_ylabel('pct_counts_mt')

plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'qc_violins_clean.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Per-patient comparison to see outtliers
fig, axs = plt.subplots(1, 3, figsize=(20, 5))

for i, (metric, log_scale) in enumerate([
    ('n_genes_by_counts', True),
    ('total_counts', True),
    ('pct_counts_mt', False),
]):
    sns.violinplot(
        x='orig.ident', y=metric, data=adata.obs,
        ax=axs[i], inner='quartile', linewidth=0.5,
    )
    if log_scale:
        axs[i].set_yscale('log')
    axs[i].set_xlabel('Patient')
    axs[i].tick_params(axis='x', rotation=90)
    axs[i].set_title(metric)

plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'qc_violins_per_patient.png', dpi=150, bbox_inches='tight')
plt.show()

## Apply filters

In [ ]:
# 1. Basic filters: min genes per cell, min cells per gene
adata = qc.filter_cells_genes(
    adata,
    min_genes=cfg.MIN_GENES_PER_CELL,
    min_cells=cfg.MIN_CELLS_PER_GENE,
)

In [ ]:
#Check if ARC is in the dataset
print(f'ARC in dataset: {"ARC" in adata.var_names}')
print(f'Total cells expressing ARC: {(adata[:, "ARC"].X > 0).sum() if "ARC" in adata.var_names else "N/A"}')

In [ ]:
# 2. Mitochondrial % filter
adata = qc.filter_by_mito(adata, max_pct=cfg.MAX_PCT_MITO)

In [ ]:
# 3. Outlier removal by total counts (per-patient MAD). Catches likely doublets. This step was removed to make sure to include cycling cells
"""
adata = qc.filter_by_counts_mad(
    adata,
    batch_key='Patient',
    n_mads=cfg.MAX_TOTAL_COUNTS_MAD,
)
"""

In [ ]:
# 4. Scrublet doublet detection, per-patient
#
# Skip in DEV_MODE if memory is tight — scrublet needs room for the simulated
# doublet matrix. In DEV_MODE on 8 GB RAM it should still work per-patient.
adata = qc.run_scrublet(
    adata,
    batch_key=cfg.BATCH_KEY,
    expected_doublet_rate=cfg.EXPECTED_DOUBLET_RATE,
    random_state=cfg.RANDOM_SEED,
)

In [ ]:
print(f"Total cells: {adata.n_obs}")
print(f"Has 'predicted_doublet' column: {'predicted_doublet' in adata.obs.columns}")
print(f"Has 'doublet_score' column: {'doublet_score' in adata.obs.columns}")
if 'doublet_score' in adata.obs.columns:
    print(f"\nDoublet score distribution:")
    print(adata.obs['doublet_score'].describe())
    print(f"\nNon-zero doublet scores: {(adata.obs['doublet_score'] > 0).sum()}")
if 'predicted_doublet' in adata.obs.columns:
    print(f"\nPredicted doublet count: {adata.obs['predicted_doublet'].sum()}")

In [ ]:
print(f"Cell count: {adata.n_obs}")
print(f"Has 'predicted_doublet': {'predicted_doublet' in adata.obs.columns}")
print(f"Has 'pct_counts_mt': {'pct_counts_mt' in adata.obs.columns}")
print(f"Patients (orig.ident): {adata.obs['orig.ident'].nunique()}")
print(f"Type of adata.X: {type(adata.X)}")
print(f"Mito % range: {adata.obs['pct_counts_mt'].min():.1f}% to {adata.obs['pct_counts_mt'].max():.1f}%")

In [ ]:
# Per-patient summary AFTER filtering
qc.summarize_qc(adata).head(30)

In [ ]:
import seaborn as sns

fig, axs = plt.subplots(1, 3, figsize=(14, 4))

sns.violinplot(y=adata.obs['n_genes_by_counts'], ax=axs[0], color='steelblue', inner='quartile')
axs[0].set_title('Genes per cell (after QC)')
axs[0].set_yscale('log')
axs[0].set_ylabel('n_genes_by_counts (log)')

sns.violinplot(y=adata.obs['total_counts'], ax=axs[1], color='steelblue', inner='quartile')
axs[1].set_title('UMI counts per cell (after QC)')
axs[1].set_yscale('log')
axs[1].set_ylabel('total_counts (log)')

sns.violinplot(y=adata.obs['pct_counts_mt'], ax=axs[2], color='steelblue', inner='quartile')
axs[2].set_title('Mitochondrial % (after QC)')
axs[2].axhline(20, color='red', linestyle='--', label='Wu cutoff (20%)')
axs[2].legend()
axs[2].set_ylabel('pct_counts_mt')

plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'qc_violins_after_clean.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'Final cell count after QC: {adata.n_obs}')
print(f'Final gene count after QC: {adata.n_vars}')
print(f'ARC still present: {"ARC" in adata.var_names}')

adata.write_h5ad(cfg.H5AD_QC, compression='gzip')
print(f'Saved: {cfg.H5AD_QC}')

---

**Next**: `03_normalization_hvg.ipynb`